# OmiGuard Lite Training Pipeline

This notebook trains the production model artifacts used by the FastAPI service.

It expects `data/model_ready_sensor_data.csv`, created by `src/manual_pull_build_dataset.py`. The dataset already contains cleaned sensor readings, engineered features, and WHO 2021 air-quality guideline screening indexes where `100` means the reading is at the WHO guideline level.

Outputs are written to `models/`:

- `risk_model.pkl`
- `anomaly_model.pkl`
- `label_encoder.pkl`
- `feature_cols.pkl`
- `metrics.txt`

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, IsolationForest
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## Configuration

The path logic works when the notebook is run from either `gas_ai_system/src` or the project root.

In [ ]:
try:
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    current = Path.cwd().resolve()
    BASE_DIR = current.parent if current.name == "src" else current
    if BASE_DIR.name != "gas_ai_system" and (BASE_DIR / "gas_ai_system").exists():
        BASE_DIR = BASE_DIR / "gas_ai_system"

DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
MODEL_READY_DATA_PATH = DATA_DIR / "model_ready_sensor_data.csv"
METRICS_PATH = MODELS_DIR / "metrics.txt"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"Model-ready dataset: {MODEL_READY_DATA_PATH}")
print(f"Models directory: {MODELS_DIR}")

## Feature Contract

These are the exact columns saved to `feature_cols.pkl` and consumed by `src/api.py`. Gas-derived AQI columns are model inputs; string columns such as `aqi_category`, `rule_based_risk_class`, and `who_dominant_pollutant` are kept for review only.

In [ ]:
TARGET_COL = "risk_class"
VALID_CLASSES = ["Safe", "Moderate", "Caution", "Dangerous"]

FEATURE_COLS = [
    "co",
    "so2",
    "no2",
    "pm1_0",
    "pm2_5",
    "pm10",
    "temperature",
    "humidity",
    "hour",
    "day_of_week",
    "total_gas_load",
    "total_pm_load",
    "co_so2_interaction",
    "gas_pm_interaction",
    "temp_humidity_index",
    "heat_stress_flag",
    "high_humidity_flag",
    "pm2_5_pm10_ratio",
    "pm1_pm2_5_ratio",
    "child_exposure_score",
    "aqi_pm2_5",
    "aqi_pm10",
    "aqi",
]

DIAGNOSTIC_COLS = [
    "co_mg_m3",
    "no2_ug_m3",
    "so2_ug_m3",
    "aqi_co",
    "aqi_no2",
    "aqi_so2",
    "aqi_category",
    "who_aqg_exceedance_count",
    "who_dominant_pollutant",
    "rule_based_risk_class",
]

REQUIRED_COLUMNS = FEATURE_COLS + [TARGET_COL]
print(f"Training features: {len(FEATURE_COLS)}")

## Load And Validate Data

The notebook fails early if the dataset still has old labels such as `low/high`, missing AQI fields, or non-numeric feature values.

In [ ]:
if not MODEL_READY_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Model-ready dataset not found: {MODEL_READY_DATA_PATH}. "
        "Run src/manual_pull_build_dataset.py first."
    )

df_raw = pd.read_csv(MODEL_READY_DATA_PATH)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

In [ ]:
missing = [col for col in REQUIRED_COLUMNS if col not in df_raw.columns]
if missing:
    raise ValueError(
        "Missing required columns in model_ready_sensor_data.csv: "
        f"{missing}. Run src/manual_pull_build_dataset.py again."
    )

available_diagnostics = [col for col in DIAGNOSTIC_COLS if col in df_raw.columns]
missing_diagnostics = [col for col in DIAGNOSTIC_COLS if col not in df_raw.columns]

print("Available diagnostics:", available_diagnostics)
if missing_diagnostics:
    print("Missing optional diagnostics:", missing_diagnostics)

missing

In [ ]:
df_model = df_raw.copy()
df_model[TARGET_COL] = df_model[TARGET_COL].astype(str).str.strip()

invalid_labels = sorted(set(df_model[TARGET_COL]) - set(VALID_CLASSES))
if invalid_labels:
    raise ValueError(
        f"Invalid {TARGET_COL} labels found: {invalid_labels}. "
        f"Expected only: {VALID_CLASSES}. "
        "This usually means the CSV was generated by an older preprocessing script."
    )

for col in FEATURE_COLS:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

before = len(df_model)
df_model = df_model.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)
after = len(df_model)

if df_model.empty:
    raise ValueError("No valid training rows remain after cleaning.")

print(f"Rows before cleaning: {before}")
print(f"Rows after cleaning: {after}")
print("Risk class distribution:")
print(df_model[TARGET_COL].value_counts())

## Data Review

These quick checks confirm that the labels and WHO screening indexes are aligned before training.

In [ ]:
summary_cols = [
    "pm2_5",
    "pm10",
    "co",
    "so2",
    "no2",
    "aqi_pm2_5",
    "aqi_pm10",
    "aqi",
]

if available_diagnostics:
    display_cols = [col for col in [
        "timestamp",
        "pm2_5",
        "pm10",
        "aqi_pm2_5",
        "aqi_pm10",
        "aqi_co",
        "aqi_no2",
        "aqi_so2",
        "aqi",
        "aqi_category",
        "who_dominant_pollutant",
        TARGET_COL,
    ] if col in df_model.columns]
    display(df_model[display_cols].tail(10))

df_model[summary_cols].describe().round(2)

In [ ]:
if "who_dominant_pollutant" in df_model.columns:
    print("Dominant pollutant distribution:")
    print(df_model["who_dominant_pollutant"].value_counts())

if "who_aqg_exceedance_count" in df_model.columns:
    print("\nWHO guideline exceedance-count distribution:")
    print(df_model["who_aqg_exceedance_count"].value_counts().sort_index())

## Train Risk Classifier

The model predicts `risk_class` from the feature contract above. Stratification is enabled only when every class has at least two examples, which keeps the notebook robust for small field datasets.

In [ ]:
X = df_model[FEATURE_COLS]
y = df_model[TARGET_COL]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

class_counts = y.value_counts()
use_stratify = len(class_counts) > 1 and class_counts.min() >= 2
stratify = y_encoded if use_stratify else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=stratify,
)

print(f"Train rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Classes: {list(label_encoder.classes_)}")

In [ ]:
risk_model = GradientBoostingClassifier(random_state=42)
risk_model.fit(X_train, y_train)

predictions = risk_model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
macro_f1 = f1_score(y_test, predictions, average="macro", zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    target_names=label_encoder.inverse_transform(np.unique(y_test)),
    zero_division=0,
))

In [ ]:
labels_in_test = label_encoder.inverse_transform(np.unique(y_test))
cm = confusion_matrix(y_test, predictions, labels=np.unique(y_test))
cm_df = pd.DataFrame(cm, index=labels_in_test, columns=labels_in_test)
cm_df

## Train Anomaly Detector

The anomaly detector is trained on the same feature matrix. The contamination value is kept conservative for this small dataset.

In [ ]:
contamination = min(0.1, max(1 / len(X), 0.05))

anomaly_model = IsolationForest(
    contamination=contamination,
    random_state=42,
)
anomaly_model.fit(X)

anomaly_flags = anomaly_model.predict(X) == -1
print(f"Contamination used: {contamination:.4f}")
print(f"Rows flagged as anomalous: {int(anomaly_flags.sum())} of {len(X)}")

## Save Artifacts

These files are what the API loads at startup.

In [ ]:
joblib.dump(risk_model, MODELS_DIR / "risk_model.pkl")
joblib.dump(anomaly_model, MODELS_DIR / "anomaly_model.pkl")
joblib.dump(label_encoder, MODELS_DIR / "label_encoder.pkl")
joblib.dump(FEATURE_COLS, MODELS_DIR / "feature_cols.pkl")

with open(METRICS_PATH, "w", encoding="utf-8") as file:
    file.write("OmiGuard Lite Model Training Metrics\n")
    file.write("====================================\n")
    file.write(f"Rows: {len(df_model)}\n")
    file.write(f"Features: {len(FEATURE_COLS)}\n")
    file.write(f"Feature columns: {FEATURE_COLS}\n")
    file.write(f"Classes: {list(label_encoder.classes_)}\n")
    file.write(f"Accuracy: {accuracy:.4f}\n")
    file.write(f"Macro F1: {macro_f1:.4f}\n")

print("Saved model artifacts:")
for path in [
    MODELS_DIR / "risk_model.pkl",
    MODELS_DIR / "anomaly_model.pkl",
    MODELS_DIR / "label_encoder.pkl",
    MODELS_DIR / "feature_cols.pkl",
    METRICS_PATH,
]:
    print(path)

## Smoke Test

Run one prediction using the latest row in the model-ready dataset. This mirrors the FastAPI request path because the model consumes the saved `FEATURE_COLS` list in this exact order.

In [ ]:
latest_row = df_model.iloc[-1:][FEATURE_COLS]

pred_encoded = risk_model.predict(latest_row)[0]
pred_label = label_encoder.inverse_transform([pred_encoded])[0]
proba = risk_model.predict_proba(latest_row)[0]
anomaly_flag = bool(anomaly_model.predict(latest_row)[0] == -1)

result = {
    "risk_class": pred_label,
    "risk_score": round(float(np.max(proba)), 4),
    "anomaly_flag": anomaly_flag,
}

result

## Operational Notes

- Regenerate `model_ready_sensor_data.csv` with `src/manual_pull_build_dataset.py` before retraining.
- Keep `FEATURE_COLS` synchronized with `src/train.py` and the `feature_cols.pkl` artifact used by `src/api.py`.
- The current dataset only contains `Safe` and `Dangerous` labels. The pipeline supports `Moderate` and `Caution`, but the model cannot learn those classes until examples exist in the data.